# TF × Regulatory Type Analysis

This notebook cross-tabulates transcription factor (TF) motif enrichment with regulatory element type (promoter vs enhancer-like) using HOMER results. The goal is to identify TFs that preferentially associate with promoters or enhancers.

**Analysis steps:**
1. Import required libraries and define file paths
2. Parse motif and annotation results for each peak set
3. Classify peaks as enhancer-like or promoter
4. Cross-tabulate TF occurrence by regulatory type

---

## 1. Import Required Libraries and Define Paths

Import the necessary Python libraries and define the paths to the HOMER motif and annotation files for each peak set.

In [1]:
# Import libraries
import pandas as pd
import os

# Define file paths for each peak set
base_dir = "HOMER/results"
peak_sets = {
    "shared": {
        "motif": os.path.join(base_dir, "shared_peaks_conservative/motifs/knownResults.txt"),
        "annot": os.path.join(base_dir, "shared_peaks_conservative/annotated_peaks.txt"),
    },
    "human_specific": {
        "motif": os.path.join(base_dir, "human_specific_peaks_conservative/motifs/knownResults.txt"),
        "annot": os.path.join(base_dir, "human_specific_peaks_conservative/annotated_peaks.txt"),
    },
    "mouse_specific": {
        "motif": os.path.join(base_dir, "mouse_specific_peaks_conservative/motifs/knownResults.txt"),
        "annot": os.path.join(base_dir, "mouse_specific_peaks_conservative/annotated_peaks.txt"),
    },
}
peak_sets

{'shared': {'motif': 'HOMER/results/shared_peaks_conservative/motifs/knownResults.txt',
  'annot': 'HOMER/results/shared_peaks_conservative/annotated_peaks.txt'},
 'human_specific': {'motif': 'HOMER/results/human_specific_peaks_conservative/motifs/knownResults.txt',
  'annot': 'HOMER/results/human_specific_peaks_conservative/annotated_peaks.txt'},
 'mouse_specific': {'motif': 'HOMER/results/mouse_specific_peaks_conservative/motifs/knownResults.txt',
  'annot': 'HOMER/results/mouse_specific_peaks_conservative/annotated_peaks.txt'}}

## 2. Parse Motif and Annotation Results

For each peak set, parse the motif and annotation files. Extract the TF motif assignments and regulatory annotation for each peak.

In [32]:
import pandas as pd
import re

def clean_homer_motif(path):
    """
    Robust HOMER knownResults.txt parser (force first N columns, ignore extras).
    """
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    if not lines:
        return pd.DataFrame()
    
    
    colnames = ["Motif Name", "Consensus", "P-value", "Log P-value", "q-value"]
    data_lines = lines[1:]
    
     
    data_lines = [l for l in data_lines if not l.startswith("Motif")]
    
   
    rows = [line.split("\t")[:5] for line in data_lines]
    
    df = pd.DataFrame(rows, columns=colnames)
    
     
    def clean_cell(x):
        x = str(x)
        if re.match(r"^\d+\.\d+\.\d+$", x):
            return ".".join(x.split(".")[:2])
        if "%" in x:
            return x.split("%")[0] + "%"
        return x
        
    df = df.applymap(clean_cell)  
    df["TF"] = df["Motif Name"].astype(str).str.split("/").str[0]
    
    return df

In [33]:
# Use the robust parser for all peak sets
motif_dfs = {k: clean_homer_motif(v['motif']) for k, v in peak_sets.items()}

# Example: show shared motif table
motif_df = motif_dfs['shared']
motif_df

/tmp/ipykernel_1833433/1878088648.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_cell)
/tmp/ipykernel_1833433/1878088648.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_cell)
/tmp/ipykernel_1833433/1878088648.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_cell)


,Motif Name,Consensus,P-value,Log P-value,q-value,TF
0,AP-1(bZIP)/ThioMac-PU.1-ChIP-Seq(GSE21512)/Homer,VTGACTCATC,1e0,0.000e+00,1.0000,AP-1(bZIP)
1,AP-2gamma(AP2)/MCF7-TFAP2C-ChIP-Seq(GSE21234)/...,SCCTSAGGSCAW,1e0,0.000e+00,1.0000,AP-2gamma(AP2)
2,AP-2alpha(AP2)/Hela-AP2alpha-ChIP-Seq(GSE31477...,ATGCCCTGAGGC,1e0,0.000e+00,1.0000,AP-2alpha(AP2)
3,Ap4(bHLH)/AML-Tfap4-ChIP-Seq(GSE45738)/Homer,NAHCAGCTGD,1e0,0.000e+00,1.0000,Ap4(bHLH)
4,"FOXA1:AR(Forkhead,NR)/LNCAP-AR-ChIP-Seq(GSE278...",AGTAAACAAAAAAGAACAND,1e0,0.000e+00,1.0000,"FOXA1:AR(Forkhead,NR)"
...,...,...,...,...,...,...
467,ZNF768(Zf)/Rajj-ZNF768-ChIP-Seq(GSE111879)/Homer,RHHCAGAGAGGB,1e0,0.000e+00,1.0000,ZNF768(Zf)
468,ZNF7(Zf)/HepG2-ZNF7.Flag-ChIP-Seq(Encode)/Homer,CTGCCWVCTTTTRTA,1e0,0.000e+00,1.0000,ZNF7(Zf)
469,ZNF91(Zf)/HEK-ZNF91.HA-ChIP-Seq(GSE162571)/Homer,RGCMGCCTYC,1e0,0.000e+00,1.0000,ZNF91(Zf)
470,ZSCAN22(Zf)/HEK293-ZSCAN22.GFP-ChIP-Seq(GSE583...,SMCAGTCWGAKGGAGGAGGC,1e0,0.000e+00,1.0000,ZSCAN22(Zf)


## 3. Cross-tabulate TF Occurrence by Regulatory Type

For each TF, count its occurrence in promoter and enhancer-like peaks. Output a table: TF, promoter, enhancer-like.

In [ ]:
# Example: Cross-tabulate for shared peak set
# If you do not have annot_df, just show the top TFs table

def tf_by_reg_type(motif_df, top_n=20):
    top_tfs = motif_df['TF'].head(top_n).tolist()
    return pd.DataFrame({'TF': top_tfs})

# Show top 20 TFs only
# motif_df should be defined already (e.g., motif_df = motif_dfs['shared'])
tf_regtype_df = tf_by_reg_type(motif_df)
tf_regtype_df

## 4. Biological Insight: TF Regulatory Preferences

This table highlights TFs that are preferentially associated with promoters or enhancers, providing insight into their regulatory roles.